# Search API health check — `deep_researcher`

Checks every web-search backend the `web_research` node can dispatch to
(`agentsq/workflows/deep_researcher/graph.py:219-265`), at three levels:

1. **Raw transport** — can we even reach the provider (auth, bot-blocking, host up)?
2. **`utils.py` wrapper** — does `duckduckgo_search` / `searxng_search` / `tavily_search` /
   `perplexity_search` return the contract `{"results": [{title, url, content, raw_content}]}`?
3. **Node-level** — run the real `web_research` node with each `search_api` value, which is
   what the graph actually executes.

Run top-to-bottom; the last cell prints a PASS/FAIL/SKIP summary table.

In [9]:
import os, sys, time, json, traceback
from pathlib import Path

# `agentsq` has no packaging metadata, so it is importable only if its parent
# directory (agent-squared/) is on sys.path. Walk up from the cwd so this works
# whether the kernel started in notebooks/, agent-squared/, or the repo root.
def agentsq_root():
    for d in [Path.cwd(), *Path.cwd().parents]:
        if (d / "agentsq" / "__init__.py").exists():
            if str(d) not in sys.path:
                sys.path.insert(0, str(d))
            return d
    raise RuntimeError(f"could not locate the agentsq package from {Path.cwd()}")

ROOT = agentsq_root()

from dotenv import load_dotenv
load_dotenv(ROOT.parent / ".env")
load_dotenv(ROOT / ".env")

QUERY = "Chicago population 2025"
TIMEOUT = 20

def key_state(name):
    v = os.environ.get(name)
    return f"set ({v[:6]}...)" if v else "MISSING"

print("root:", ROOT)
for k in ["OPENAI_API_KEY", "TAVILY_API_KEY", "PERPLEXITY_API_KEY", "SEARXNG_URL", "SEARCH_API"]:
    print(f"  {k:20s} {key_state(k)}")
print()
print("NOTE: Configuration.from_runnable_config reads os.environ[FIELD.upper()] BEFORE the")
print("      runnable config, so a stray SEARCH_API env var silently overrides the adapter.")

root: /home/jsh27/agent-monitor/agent-squared
  OPENAI_API_KEY       set (sk-pro...)
  TAVILY_API_KEY       MISSING
  PERPLEXITY_API_KEY   MISSING
  SEARXNG_URL          MISSING
  SEARCH_API           MISSING

NOTE: Configuration.from_runnable_config reads os.environ[FIELD.upper()] BEFORE the
      runnable config, so a stray SEARCH_API env var silently overrides the adapter.


## 1. Raw transport probes

These bypass `utils.py` entirely so a failure can be attributed to the provider, not our code.

In [10]:
import httpx

RESULTS = {}   # backend -> dict of stage -> (ok, detail)

def record(backend, stage, ok, detail):
    RESULTS.setdefault(backend, {})[stage] = (ok, detail)
    flag = {True: "PASS", False: "FAIL", None: "SKIP"}[ok]
    print(f"[{flag}] {backend:12s} {stage:10s} {detail}")

In [11]:
# --- DuckDuckGo raw: are we being served the bot-challenge page? ---
UA = ("Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) "
      "Chrome/124.0 Safari/537.36")
try:
    r = httpx.post("https://html.duckduckgo.com/html/", data={"q": QUERY},
                   headers={"User-Agent": UA}, timeout=TIMEOUT, follow_redirects=True)
    body = r.text
    blocked = ("anomaly" in body.lower()) or ("unfortunately, bots" in body.lower())
    n_hits = body.count("result__a")
    detail = f"HTTP {r.status_code}, {len(body)}B, result links={n_hits}, bot-challenge={blocked}"
    record("duckduckgo", "raw", (not blocked and n_hits > 0), detail)
except Exception as e:
    record("duckduckgo", "raw", False, f"{type(e).__name__}: {e}")

[FAIL] duckduckgo   raw        HTTP 202, 14313B, result links=0, bot-challenge=True


In [12]:
# --- Tavily raw: tavily-python >=0.8 works keyless; a key raises the rate limit ---
try:
    from tavily import TavilyClient
    c = TavilyClient()
    mode = "keyless" if getattr(c, "_keyless", c.api_key is None) else "api-key"
    r = c.search(QUERY, max_results=2)
    record("tavily", "raw", len(r.get("results", [])) > 0,
           f"mode={mode}, results={len(r.get('results', []))}")
except Exception as e:
    record("tavily", "raw", False, f"{type(e).__name__}: {str(e)[:160]}")

[PASS] tavily       raw        mode=keyless, results=2


In [13]:
# --- Perplexity raw: needs PERPLEXITY_API_KEY ---
if not os.environ.get("PERPLEXITY_API_KEY"):
    record("perplexity", "raw", None, "PERPLEXITY_API_KEY not set")
else:
    try:
        r = httpx.post("https://api.perplexity.ai/chat/completions",
                       headers={"Authorization": f"Bearer {os.environ['PERPLEXITY_API_KEY']}",
                                "content-type": "application/json"},
                       json={"model": "sonar-pro",
                             "messages": [{"role": "user", "content": QUERY}]},
                       timeout=TIMEOUT)
        record("perplexity", "raw", r.status_code == 200, f"HTTP {r.status_code}")
    except Exception as e:
        record("perplexity", "raw", False, f"{type(e).__name__}: {str(e)[:160]}")

[SKIP] perplexity   raw        PERPLEXITY_API_KEY not set


In [14]:
# --- SearXNG raw: needs an instance whose settings.yml enables the JSON format ---
host = os.environ.get("SEARXNG_URL", "http://localhost:8888")
try:
    r = httpx.get(host.rstrip("/") + "/search",
                  params={"q": QUERY, "format": "json"},
                  headers={"User-Agent": UA}, timeout=TIMEOUT, follow_redirects=True)
    ct = r.headers.get("content-type", "")
    if "json" in ct:
        n = len(r.json().get("results", []))
        record("searxng", "raw", n > 0, f"{host} HTTP {r.status_code}, results={n}")
    else:
        record("searxng", "raw", False,
               f"{host} HTTP {r.status_code}, content-type={ct[:30]} "
               f"(JSON format disabled -> add 'json' to search.formats in settings.yml)")
except Exception as e:
    record("searxng", "raw", False, f"{host} unreachable: {type(e).__name__}: {str(e)[:120]}")

[FAIL] searxng      raw        http://localhost:8888 unreachable: ConnectError: [Errno 111] Connection refused


## 2. `utils.py` wrapper contract

Every backend must return `{"results": [{"title", "url", "content", "raw_content"}]}`.
`duckduckgo_search` swallows *all* exceptions into `{"results": []}`
(`utils.py:216-219`), so an empty list here means "silently broken", not "no matches".

In [15]:
agentsq_root()   # safe to re-run; makes this cell work even if run out of order

from agentsq.workflows.deep_researcher.utils import (
    duckduckgo_search, searxng_search, tavily_search, perplexity_search,
    deduplicate_and_format_sources, format_sources,
)

REQUIRED = {"title", "url", "content", "raw_content"}

def check_contract(backend, fn, *args, **kwargs):
    t0 = time.time()
    try:
        out = fn(*args, **kwargs)
    except Exception as e:
        record(backend, "wrapper", False, f"raised {type(e).__name__}: {str(e)[:160]}")
        return None
    dt = time.time() - t0
    results = out.get("results", []) if isinstance(out, dict) else None
    if results is None:
        record(backend, "wrapper", False, f"bad shape: {type(out).__name__}")
        return out
    if not results:
        record(backend, "wrapper", False, f"0 results in {dt:.1f}s (errors are swallowed)")
        return out
    missing = REQUIRED - set(results[0])
    empty_raw = sum(1 for r in results if not r.get("raw_content"))
    record(backend, "wrapper", not missing,
           f"{len(results)} results in {dt:.1f}s, missing_keys={sorted(missing) or 'none'}, "
           f"empty raw_content={empty_raw}/{len(results)}")
    return out

In [16]:
# fetch_full_page mirrors Configuration's default (True): ddg/searxng then re-fetch each URL
# through utils.fetch_raw_content (httpx + markdownify, 10s timeout).
ddg_out    = check_contract("duckduckgo", duckduckgo_search, QUERY, max_results=3, fetch_full_page=True)
tavily_out = check_contract("tavily", tavily_search, QUERY, fetch_full_page=True, max_results=3)

if os.environ.get("PERPLEXITY_API_KEY"):
    ppx_out = check_contract("perplexity", perplexity_search, QUERY, 0)
else:
    ppx_out = None
    record("perplexity", "wrapper", None, "no API key")

if RESULTS.get("searxng", {}).get("raw", (False,))[0]:
    searx_out = check_contract("searxng", searxng_search, QUERY, max_results=3, fetch_full_page=True)
else:
    searx_out = None
    record("searxng", "wrapper", None, "no reachable JSON-enabled instance")

/home/jsh27/agent-monitor/agent-squared/agentsq/workflows/deep_researcher/utils.py:189: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


[FAIL] duckduckgo   wrapper    0 results in 0.4s (errors are swallowed)
[PASS] tavily       wrapper    3 results in 0.3s, missing_keys=none, empty raw_content=0/3
[SKIP] perplexity   wrapper    no API key
[SKIP] searxng      wrapper    no reachable JSON-enabled instance


In [17]:
# What the LLM actually receives, and what an empty search degrades to.
for name, out in [("duckduckgo", ddg_out), ("tavily", tavily_out),
                  ("perplexity", ppx_out), ("searxng", searx_out)]:
    if out is None:
        continue
    formatted = deduplicate_and_format_sources(out, max_tokens_per_source=1000, fetch_full_page=True)
    print(f"--- {name}: format_sources ---")
    print(format_sources(out) or "(nothing)")
    print(f"--- {name}: context passed to summarize_sources ({len(formatted)} chars) ---")
    print(formatted[:600])
    print()

--- duckduckgo: format_sources ---
(nothing)
--- duckduckgo: context passed to summarize_sources (8 chars) ---
Sources:

--- tavily: format_sources ---
* Chicago's population grew to 2.73M people in 2025, per ... : https://www.fox32chicago.com/news/chicago-population-grew-2025-census-estimate
* Where does Chicago's population stand with other US cities? : https://www.nbcchicago.com/news/local/where-does-chicagos-population-stand-compared-to-other-big-cities-the-latest-census-data-reveals-unexpected-change/3936544
* U.S. Census Bureau QuickFacts: Chicago city, Illinois : https://www.census.gov/quickfacts/fact/table/chicagocityillinois/PST045225
--- tavily: context passed to summarize_sources (16338 chars) ---
Sources:

Source: Chicago's population grew to 2.73M people in 2025, per ...
===
URL: https://www.fox32chicago.com/news/chicago-population-grew-2025-census-estimate
===
Most relevant content from source: CHICAGO - The City of Chicago’s population grew to more than 2.73 million peop

## 3. Node-level check

Runs the real `web_research` node (`graph.py:201`) for each `search_api`, i.e. the exact path
the graph takes. A backend that passes here is usable for a benchmark baseline.

In [18]:
agentsq_root()   # safe to re-run; makes this cell work even if run out of order

from agentsq.workflows.deep_researcher.graph import web_research
from agentsq.workflows.deep_researcher.state import SummaryState

def run_node(search_api):
    state = SummaryState(research_topic=QUERY, search_query=QUERY, research_loop_count=0)
    config = {"configurable": {"search_api": search_api, "fetch_full_page": True}}
    try:
        out = web_research(state, config)
    except Exception as e:
        record(search_api, "node", False, f"raised {type(e).__name__}: {str(e)[:160]}")
        return
    text = out["web_research_results"][0]
    sources = out["sources_gathered"][0]
    ok = bool(sources.strip()) and len(text) > len("Sources:")
    record(search_api, "node", ok, f"context={len(text)} chars, sources={len(sources.splitlines())}")

for api in ["duckduckgo", "tavily", "perplexity", "searxng"]:
    skip = RESULTS.get(api, {}).get("wrapper", (None,))[0] is None
    if skip:
        record(api, "node", None, "skipped (wrapper unavailable)")
    else:
        run_node(api)

/home/jsh27/agent-monitor/agent-squared/agentsq/workflows/deep_researcher/utils.py:189: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


[FAIL] duckduckgo   node       context=8 chars, sources=0
[PASS] tavily       node       context=5466 chars, sources=1
[SKIP] perplexity   node       skipped (wrapper unavailable)
[SKIP] searxng      node       skipped (wrapper unavailable)


## 4. Reliability probe

A backend that answers *once* is not a baseline. DuckDuckGo in particular is rate-limited by
bot detection: `DDGS.text()` returns an empty list rather than raising, and
`duckduckgo_search` turns any exception into `{"results": []}` too, so a throttled run looks
exactly like "the web had no answer". This cell fires several distinct queries back-to-back
and reports the hit rate.

In [11]:
PROBE_QUERIES = [
    "Chicago population 2025",
    "langgraph conditional edges documentation",
    "quantum error correction milestone 2025",
    "who won the 2024 world series",
    "transformer attention mechanism explained",
    "US inflation rate september 2025",
]

def reliability(backend, fn, **kwargs):
    hits = []
    for q in PROBE_QUERIES:
        try:
            n = len(fn(q, **kwargs).get("results", []))
        except Exception as e:
            n = f"ERR:{type(e).__name__}"
        hits.append(n)
        time.sleep(1)
    ok = sum(1 for h in hits if isinstance(h, int) and h > 0)
    print(f"{backend:12s} {ok}/{len(PROBE_QUERIES)} queries returned results   {hits}")
    return ok / len(PROBE_QUERIES)

RELIABILITY = {}
RELIABILITY["duckduckgo"] = reliability("duckduckgo", duckduckgo_search, max_results=3, fetch_full_page=False)
RELIABILITY["tavily"] = reliability("tavily", tavily_search, max_results=3, fetch_full_page=False)
if os.environ.get("PERPLEXITY_API_KEY"):
    RELIABILITY["perplexity"] = reliability("perplexity", perplexity_search)
if RESULTS.get("searxng", {}).get("raw", (False,))[0]:
    RELIABILITY["searxng"] = reliability("searxng", searxng_search, max_results=3, fetch_full_page=False)

/home/jsh27/agent-monitor/agent-squared/agentsq/workflows/deep_researcher/utils.py:189: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


/home/jsh27/agent-monitor/agent-squared/agentsq/workflows/deep_researcher/utils.py:189: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


/home/jsh27/agent-monitor/agent-squared/agentsq/workflows/deep_researcher/utils.py:189: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


/home/jsh27/agent-monitor/agent-squared/agentsq/workflows/deep_researcher/utils.py:189: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


/home/jsh27/agent-monitor/agent-squared/agentsq/workflows/deep_researcher/utils.py:189: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


/home/jsh27/agent-monitor/agent-squared/agentsq/workflows/deep_researcher/utils.py:189: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


duckduckgo   2/6 queries returned results   [0, 0, 0, 0, 3, 3]


tavily       6/6 queries returned results   [3, 3, 3, 3, 3, 3]


## 4. Summary

In [12]:
FLAG = {True: "PASS", False: "FAIL", None: "SKIP"}
print(f"{'backend':12s} {'raw':6s} {'wrapper':8s} {'node':6s} {'hit rate':9s}")
print("-" * 46)
for backend in ["duckduckgo", "tavily", "perplexity", "searxng"]:
    row = RESULTS.get(backend, {})
    print(f"{backend:12s} "
          f"{FLAG[row.get('raw', (None,))[0]]:6s} "
          f"{FLAG[row.get('wrapper', (None,))[0]]:8s} "
          f"{FLAG[row.get('node', (None,))[0]]:6s} "
          f"{RELIABILITY.get(backend, float('nan')):.0%}")
print()
usable = [b for b, row in RESULTS.items() if row.get("node", (None,))[0]]
print("usable as a healthy baseline:", usable or "NONE")

backend      raw    wrapper  node   hit rate 
----------------------------------------------
duckduckgo   FAIL   FAIL     FAIL   33%
tavily       PASS   PASS     PASS   100%
perplexity   SKIP   SKIP     SKIP   nan%
searxng      FAIL   SKIP     SKIP   nan%

usable as a healthy baseline: ['tavily']
